# Runoff Analysis – Live Data

This notebook pulls **real-time** streamflow and precipitation data from the internet and refreshes the analysis every time it is run.  
Re-run all cells (`Kernel → Restart & Run All`) at any time to get the latest numbers.

**Data sources used here (all free / no API key required):**
* [USGS National Water Information System](https://waterservices.usgs.gov/) – daily mean streamflow / discharge
* [NOAA National Weather Service](https://api.weather.gov/) – recent observed precipitation

Add your own URLs in the **Configuration** section below.

## 1  Imports

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta, timezone
from io import StringIO
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print(f"Notebook run at: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2  Configuration

Edit the values in this cell to point to your own data sources or adjust the analysis window.

In [ ]:
# ── Analysis window ──────────────────────────────────────────────────────────
DAYS_BACK = 30          # how many days of history to retrieve

# ── USGS streamflow stations ──────────────────────────────────────────────────
# Find station IDs at https://waterdata.usgs.gov/nwis/rt
USGS_STATIONS = {
    "Eagle River at Avon, CO": "09067005",
    "Eagle River bl Milk Creek nr Wolcott, CO": "09070000",
}

# ── NOAA weather station (used for precipitation) ────────────────────────────
# Find station IDs at https://www.weather.gov/
# Example: KEGS = Eagle, CO
NOAA_STATION_ID = "KEGS"    # ICAO airport code near your area of interest

# ── Your own custom data URLs ─────────────────────────────────────────────────
# Add any URLs that return CSV, JSON, or plain-text data.
# Each entry: { "label": "...", "url": "...", "format": "csv" | "json" }
CUSTOM_DATA_SOURCES = [
    # Example (uncomment and edit):
    # {"label": "My gauge data", "url": "https://example.com/data.csv", "format": "csv"},
]

# ── Runoff analysis thresholds ────────────────────────────────────────────────
HIGH_FLOW_PERCENTILE   = 90   # flag flows above this percentile as high
LOW_FLOW_PERCENTILE    = 10   # flag flows below this percentile as low

# ── Request timeout ──────────────────────────────────────────────────────────
REQUEST_TIMEOUT = 30  # seconds; increase for slow connections

## 3  Data Fetching

### 3a  USGS Streamflow

In [ ]:
def fetch_usgs_streamflow(station_id: str, days_back: int = 30) -> pd.DataFrame:
    """Fetch daily mean discharge (cfs) from USGS NWIS."""
    end_dt   = datetime.now(timezone.utc).date()
    start_dt = end_dt - timedelta(days=days_back)

    url = (
        "https://waterservices.usgs.gov/nwis/dv/"
        f"?format=json&sites={station_id}"
        f"&startDT={start_dt}&endDT={end_dt}"
        "&parameterCd=00060"          # 00060 = discharge, ft³/s
        "&statCd=00003"               # 00003 = mean
        "&siteStatus=all"
    )

    resp = requests.get(url, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()

    try:
        time_series = data["value"]["timeSeries"][0]["values"][0]["value"]
    except (KeyError, IndexError):
        print(f"  ⚠  No data returned for station {station_id}")
        return pd.DataFrame()

    df = pd.DataFrame(time_series)
    df["datetime"] = pd.to_datetime(df["dateTime"]).dt.tz_localize(None)
    df["discharge_cfs"] = pd.to_numeric(df["value"], errors="coerce")
    df = df[["datetime", "discharge_cfs"]].dropna()
    return df.set_index("datetime").sort_index()


# Fetch data for all configured stations
usgs_data = {}
for name, sid in USGS_STATIONS.items():
    print(f"Fetching USGS station: {name} ({sid}) ...")
    try:
        usgs_data[name] = fetch_usgs_streamflow(sid, DAYS_BACK)
        if not usgs_data[name].empty:
            latest = usgs_data[name]
            print(f"  ✓  {len(latest)} records  |  "
                  f"latest: {latest.index[-1].date()}  "
                  f"({latest['discharge_cfs'].iloc[-1]:.1f} cfs)")
    except Exception as exc:
        print(f"  ✗  Could not fetch data: {exc}")
        usgs_data[name] = pd.DataFrame()

print("Done.")

### 3b  NOAA Precipitation

In [ ]:
def fetch_noaa_precipitation(station_id: str) -> pd.DataFrame:
    """
    Fetch recent hourly precipitation observations from the NWS API.
    station_id should be an ICAO code, e.g. 'KEGS' for Eagle, CO.
    """
    url = f"https://api.weather.gov/stations/{station_id}/observations"
    headers = {"User-Agent": "ValRunoffAnalysis/1.0 (github.com/sjnovak3/ValRunoffAnalysis)"}

    resp = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
    if resp.status_code != 200:
        print(f"  ⚠  NOAA API returned {resp.status_code} for station {station_id}")
        return pd.DataFrame()

    features = resp.json().get("features", [])
    if not features:
        print(f"  ⚠  No observations for station {station_id}")
        return pd.DataFrame()

    rows = []
    for f in features:
        props = f.get("properties", {})
        ts    = props.get("timestamp")
        p_val = (props.get("precipitationLastHour") or {}).get("value")
        rows.append({"datetime": ts, "precip_mm": p_val})

    df = pd.DataFrame(rows)
    df["datetime"] = pd.to_datetime(df["datetime"]).dt.tz_localize(None)
    df["precip_mm"] = pd.to_numeric(df["precip_mm"], errors="coerce")
    return df.dropna(subset=["datetime"]).set_index("datetime").sort_index()


print(f"Fetching NOAA precipitation for station: {NOAA_STATION_ID} ...")
try:
    precip_df = fetch_noaa_precipitation(NOAA_STATION_ID)
    if not precip_df.empty:
        valid = precip_df["precip_mm"].dropna()
        print(f"  ✓  {len(precip_df)} observations  |  "
              f"latest: {precip_df.index[-1].strftime('%Y-%m-%d %H:%M')}")
        if not valid.empty:
            print(f"     total precip (available records): {valid.sum():.1f} mm")
except Exception as exc:
    print(f"  ✗  Could not fetch precipitation data: {exc}")
    precip_df = pd.DataFrame()

print("Done.")

### 3c  Custom Data Sources

In [ ]:
def fetch_custom_source(source: dict) -> pd.DataFrame:
    """
    Generic loader for CSV or JSON URLs supplied by the user.
    CSV: must contain a date/time column and at least one numeric column.
    JSON: top-level list of records, or an object with a list under a 'data' key.
    """
    fmt  = source.get("format", "csv").lower()
    resp = requests.get(source["url"], timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()

    if fmt == "csv":
        df = pd.read_csv(StringIO(resp.text))
    elif fmt == "json":
        payload = resp.json()
        if isinstance(payload, list):
            df = pd.DataFrame(payload)
        elif isinstance(payload, dict):
            df = pd.DataFrame(payload.get("data", payload))
        else:
            raise ValueError(f"Unsupported JSON shape for {source['label']}")
    else:
        raise ValueError(f"Unsupported format '{fmt}' for {source['label']}")

    return df


custom_data = {}
for src in CUSTOM_DATA_SOURCES:
    print(f"Fetching custom source: {src['label']} ...")
    try:
        custom_data[src["label"]] = fetch_custom_source(src)
        print(f"  ✓  {len(custom_data[src['label']])} rows")
    except Exception as exc:
        print(f"  ✗  Error: {exc}")

if not CUSTOM_DATA_SOURCES:
    print("No custom data sources configured (see Configuration section to add your URLs).")

## 4  Runoff Analysis

In [ ]:
def runoff_statistics(df: pd.DataFrame, col: str = "discharge_cfs") -> dict:
    """Compute key runoff statistics for a streamflow time series."""
    s = df[col].dropna()
    if s.empty:
        return {}
    return {
        "count"         : int(s.count()),
        "mean_cfs"      : round(float(s.mean()), 2),
        "median_cfs"    : round(float(s.median()), 2),
        "min_cfs"       : round(float(s.min()), 2),
        "max_cfs"       : round(float(s.max()), 2),
        "std_cfs"       : round(float(s.std()), 2),
        "cv_pct"        : round(float(s.std() / s.mean() * 100), 1) if s.mean() else None,
        f"p{HIGH_FLOW_PERCENTILE}_cfs" : round(float(np.percentile(s, HIGH_FLOW_PERCENTILE)), 2),
        f"p{LOW_FLOW_PERCENTILE}_cfs"  : round(float(np.percentile(s, LOW_FLOW_PERCENTILE)),  2),
        "high_flow_days": int((s > np.percentile(s, HIGH_FLOW_PERCENTILE)).sum()),
        "low_flow_days" : int((s < np.percentile(s, LOW_FLOW_PERCENTILE)).sum()),
        "latest_value"  : round(float(s.iloc[-1]), 2),
        "latest_date"   : df.index[-1].strftime("%Y-%m-%d"),
    }


stats_records = []
for name, df in usgs_data.items():
    if df.empty:
        continue
    st = runoff_statistics(df)
    st["station"] = name
    stats_records.append(st)

if stats_records:
    stats_df = pd.DataFrame(stats_records).set_index("station")
    print(f"=== Runoff Statistics (last {DAYS_BACK} days) ===")
    display(stats_df.T)
else:
    print("No streamflow data available for statistics.")

## 5  Visualizations

### 5a  Streamflow Time Series

In [ ]:
non_empty = {n: d for n, d in usgs_data.items() if not d.empty}

if non_empty:
    fig, axes = plt.subplots(len(non_empty), 1,
                             figsize=(14, 4 * len(non_empty)),
                             sharex=False)
    if len(non_empty) == 1:
        axes = [axes]

    for ax, (name, df) in zip(axes, non_empty.items()):
        s            = df["discharge_cfs"]
        high_thresh  = float(np.percentile(s, HIGH_FLOW_PERCENTILE))
        low_thresh   = float(np.percentile(s, LOW_FLOW_PERCENTILE))

        ax.fill_between(s.index, s, alpha=0.25, color="steelblue")
        ax.plot(s.index, s, color="steelblue", lw=1.5, label="Discharge (cfs)")
        ax.axhline(float(s.mean()), color="green",  ls="--", lw=1, label=f"Mean ({s.mean():.1f} cfs)")
        ax.axhline(high_thresh,     color="red",    ls=":",  lw=1, label=f"P{HIGH_FLOW_PERCENTILE} ({high_thresh:.1f} cfs)")
        ax.axhline(low_thresh,      color="orange", ls=":",  lw=1, label=f"P{LOW_FLOW_PERCENTILE} ({low_thresh:.1f} cfs)")

        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
        ax.set_ylabel("Discharge (cfs)")
        ax.set_title(name)
        ax.legend(loc="upper right", fontsize=8)

    fig.suptitle(f"Streamflow – last {DAYS_BACK} days", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No streamflow data to plot.")

### 5b  Precipitation (last 48 h)

In [ ]:
if not precip_df.empty:
    recent_precip = precip_df.last("48h").dropna(subset=["precip_mm"])

    if not recent_precip.empty:
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.bar(recent_precip.index, recent_precip["precip_mm"],
               width=0.04, color="cornflowerblue", edgecolor="navy", alpha=0.8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d %H:%M"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
        ax.set_ylabel("Precipitation (mm / hour)")
        ax.set_title(f"Hourly Precipitation – {NOAA_STATION_ID} (last 48 h)")
        plt.tight_layout()
        plt.show()
        print(f"Total precipitation (last 48 h): {recent_precip['precip_mm'].sum():.1f} mm")
    else:
        print("No precipitation data in the last 48 hours.")
else:
    print("No precipitation data available.")

### 5c  Rolling 7-day Average Flow

In [ ]:
if non_empty:
    fig, ax = plt.subplots(figsize=(14, 5))
    colors  = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for i, (name, df) in enumerate(non_empty.items()):
        s   = df["discharge_cfs"]
        r7  = s.rolling(7, min_periods=1).mean()
        c   = colors[i % len(colors)]
        ax.plot(s.index,  s,  color=c, alpha=0.3, lw=1)
        ax.plot(r7.index, r7, color=c, lw=2.5, label=f"{name} (7-day avg)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
    ax.set_ylabel("Discharge (cfs)")
    ax.set_title("Rolling 7-day Average Streamflow")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No streamflow data to plot.")

### 5d  Flow Duration Curve

In [ ]:
if non_empty:
    fig, ax = plt.subplots(figsize=(10, 5))

    for name, df in non_empty.items():
        s      = df["discharge_cfs"].dropna().sort_values(ascending=False)
        exceed = np.linspace(0, 100, len(s))
        ax.semilogy(exceed, s.values, lw=2, label=name)

    ax.set_xlabel("Exceedance probability (%)")
    ax.set_ylabel("Discharge (cfs, log scale)")
    ax.set_title("Flow Duration Curve")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No streamflow data to plot.")

### 5e  Custom Data Preview

In [ ]:
for label, df in custom_data.items():
    print(f"── {label} ──")
    display(df.head(10))
    print(f"Shape: {df.shape}\n")

if not custom_data:
    print("No custom data sources were loaded (see Configuration section to add your URLs).")

## 6  Daily Refresh Helper

Run the cell below to print the timestamp and a reminder of how to automate this notebook.

In [ ]:
print("=" * 60)
print(f"  Analysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)
print()
print("To schedule daily automatic refresh you can use:")
print()
print("  Option A – nbconvert (cron / Task Scheduler):")
print("    jupyter nbconvert --to notebook --execute runoff_analysis.ipynb")
print()
print("  Option B – GitHub Actions workflow:")
print("    See .github/workflows/daily_refresh.yml (create one to run nbconvert)")
print()
print("  Option C – Papermill (parameterised execution):")
print("    papermill runoff_analysis.ipynb runoff_analysis_output.ipynb")